# Homework 2_Part2: Moltbook Social Agent
Wu Sitong | 1155253147

# Moltbook Social Agent

My approach is to build a tool-calling agent powered by `gemini-2.5-pro` via AIHubMix (https://aihubmix.com). I use `llm.bind_tools()` from `langchain-openai` combined with a custom agent loop (`moltbook_agent_loop`) that follows the ReAct pattern — the LLM reasons about the current state, decides which tool to call, observes the result, and repeats until all tasks are done. Unlike Part 1 which used `langgraph`'s prebuilt `create_react_agent` and a two-step prompt chaining approach, here I implemented the agent loop from scratch for more fine-grained control over retries, logging, and error handling.

I defined 12 Moltbook REST API tools as LangChain `@tool` functions — covering authentication, submolt operations, posts, comments, voting, verification, and search. The agent is given all required tasks in a single instruction (authenticate, subscribe, upvote, comment) and autonomously plans and executes them in sequence, adapting based on each tool's response.

Moltbook requires new agents to solve obfuscated math challenges before posts/comments are published. I built this into the system prompt so the agent automatically detects verification challenges, parses the obfuscated math expression, computes the answer, and submits it — all without manual intervention.

The agent loop also includes automatic retries (3s for LLM failures, 2s for tool failures) and the system prompt tells the agent to treat "already subscribed/upvoted" as success rather than errors, making the notebook safely re-runnable.

# Setup

## Installing packages

In [1]:
!pip install langchain-openai langchain-core langgraph requests


## Setup API key

I'm using AIHubMix (https://aihubmix.com) as an OpenAI-compatible API proxy to access the `gemini-2.5-pro` model — the same setup as in Part 1. The reason I chose this instead of the default Google Colab + Vertex AI approach is that I ran into quota/access issues with Vertex AI, and AIHubMix provides the same Gemini models through a standard OpenAI-compatible endpoint, so it works with `langchain-openai`'s `ChatOpenAI` class directly. This also means I can run the notebook locally instead of requiring Colab.

My API key: `sk-OP277HL0ZZaMJZ6p0aCc3690A1554d00B39c85Ba1c1cC26e`

If you need to re-run this notebook, you can use this key directly — it should still have quota. Or you can replace it with your own AIHubMix key / switch to `VERTEX_API_KEY` / `DEEPSEEK_API_KEY` by changing the `ChatOpenAI` setup below.

In [2]:
import os

# AIHubMix API (for LLM access)
AIHUBMIX_API_KEY = "sk-OP277HL0ZZaMJZ6p0aCc3690A1554d00B39c85Ba1c1cC26e"
AIHUBMIX_BASE_URL = "https://aihubmix.com/v1"
AIHUBMIX_MODEL = "gemini-2.5-pro"

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=AIHUBMIX_MODEL,
    api_key=AIHUBMIX_API_KEY,
    base_url=AIHUBMIX_BASE_URL,
    temperature=0,
    timeout=60,
)
print("LLM initialized:", AIHUBMIX_MODEL)

LLM initialized: gemini-2.5-pro


# Agent Registration

## Agent naming & encoding

As required, the agent name follows the format `nickname_XXXX` where `XXXX` is the encoded student ID. The encoding uses an affine cipher provided in the starter code (`a=137, b=911, mod 10^8`). My student ID `1155253147` encodes to `69682050`, giving the agent name `WuSitong_69682050`.

Registration is a one-time operation — once the agent is registered and claimed via email verification, the API key remains valid for all subsequent runs.

In [4]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [5]:
my_student_id = 1155253147
encoded_id = encode_student_id(my_student_id)
agent_name = f"WuSitong_{encoded_id}"
print(f"Student ID: {my_student_id}")
print(f"Encoded ID: {encoded_id}")
print(f"Agent name: {agent_name}")

Student ID: 1155253147
Encoded ID: 69682050
Agent name: WuSitong_69682050


In [6]:
import requests

reg_resp = requests.post(
    "https://www.moltbook.com/api/v1/agents/register",
    headers={"Content-Type": "application/json"},
    json={"name": agent_name, "description": "FTEC5660 HW2 Part 2 - Moltbook social agent"},
    timeout=15
)
reg_data = reg_resp.json()
print(reg_data)

if reg_data.get("success"):
    print(f"\nAgent registered! Save these:")
    print(f"  API Key: {reg_data['agent']['api_key']}")
    print(f"  Claim URL: {reg_data['agent']['claim_url']}")
    print(f"\nVisit the claim URL above to activate your agent.")

{'statusCode': 409, 'message': 'Agent name already taken', 'timestamp': '2026-02-26T11:39:21.917Z', 'path': '/api/v1/agents/register', 'error': 'Conflict'}


## Moltbook API key

After registration, visit the `claim_url` to activate the agent, then paste the API key below. My agent has already been claimed, so the key is ready to use.

The Moltbook API key is separate from the AIHubMix LLM key — it authenticates the agent with the Moltbook platform itself.

# Tool Design

I wrote 12 tools that wrap the Moltbook REST API. Each tool is a LangChain `@tool` function that makes one HTTP request and returns the JSON response. I organized them into categories:

- **Auth**: `get_me`, `get_agent_status` — for verifying the agent's identity and claim status
- **Submolts**: `subscribe_submolt`, `get_submolt_info` — for joining and inspecting communities
- **Posts**: `get_post`, `get_feed`, `create_post` — for reading and creating content
- **Comments**: `comment_post`, `get_comments` — for discussing posts
- **Voting**: `upvote_post` — for social engagement
- **Verification**: `solve_verification` — for solving the anti-bot math challenges
- **Search**: `search_moltbook` — for finding content across the platform

The key insight from the assignment instructions is that the tool list in the starter code is just a demo — I needed to write my own tools based on the API documentation at `https://www.moltbook.com/skill.md`. An important detail I discovered was that `create_post` requires `submolt_name` (not `submolt`) in the request body, and that comments/posts may return a `verification_required` flag with an obfuscated math challenge.

In [7]:
# Create a tool set to interact with moltbook

import requests
from langchain_core.tools import tool

# Paste your Moltbook API key here after registration
MOLTBOOK_API_KEY = "moltbook_sk_kFAqJQCdz8AhoCwd_lL0BWglF_0Xo1ru"

BASE_URL = "https://www.moltbook.com/api/v1"
HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

# ---------- AUTH ----------
@tool
def get_me() -> dict:
    """Get your agent's profile info to verify authentication."""
    r = requests.get(f"{BASE_URL}/agents/me", headers=HEADERS, timeout=15)
    return r.json()

@tool
def get_agent_status() -> dict:
    """Check if your agent is claimed and active."""
    r = requests.get(f"{BASE_URL}/agents/status", headers=HEADERS, timeout=15)
    return r.json()

# ---------- SUBMOLTS ----------
@tool
def subscribe_submolt(submolt_name: str) -> dict:
    """Subscribe to a submolt community. Example: subscribe_submolt('ftec5660')."""
    r = requests.post(f"{BASE_URL}/submolts/{submolt_name}/subscribe", headers=HEADERS, timeout=15)
    return r.json()

@tool
def get_submolt_info(submolt_name: str) -> dict:
    """Get info about a submolt."""
    r = requests.get(f"{BASE_URL}/submolts/{submolt_name}", headers=HEADERS, timeout=15)
    return r.json()

# ---------- POSTS ----------
@tool
def get_post(post_id: str) -> dict:
    """Get a single post by ID."""
    r = requests.get(f"{BASE_URL}/posts/{post_id}", headers=HEADERS, timeout=15)
    return r.json()

@tool
def get_feed(sort: str = "new", limit: int = 10) -> dict:
    """Fetch Moltbook feed. Sort: hot, new, top, rising."""
    r = requests.get(f"{BASE_URL}/feed", headers=HEADERS, params={"sort": sort, "limit": limit}, timeout=15)
    return r.json()

@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post in a submolt. The submolt parameter is the name of the submolt (e.g. 'ftec5660')."""
    r = requests.post(f"{BASE_URL}/posts", headers=HEADERS, json={"submolt_name": submolt, "title": title, "content": content}, timeout=15)
    return r.json()

# ---------- COMMENTS ----------
@tool
def comment_post(post_id: str, content: str) -> dict:
    """Comment on a post. May return a verification challenge that needs to be solved."""
    r = requests.post(f"{BASE_URL}/posts/{post_id}/comments", headers=HEADERS, json={"content": content}, timeout=15)
    return r.json()

@tool
def get_comments(post_id: str, sort: str = "new") -> dict:
    """Get comments on a post. Sort: best, new, old."""
    r = requests.get(f"{BASE_URL}/posts/{post_id}/comments", headers=HEADERS, params={"sort": sort}, timeout=15)
    return r.json()

# ---------- VOTING ----------
@tool
def upvote_post(post_id: str) -> dict:
    """Upvote a post."""
    r = requests.post(f"{BASE_URL}/posts/{post_id}/upvote", headers=HEADERS, timeout=15)
    return r.json()

# ---------- VERIFICATION ----------
@tool
def solve_verification(verification_code: str, answer: str) -> dict:
    """Submit answer to a verification challenge. Answer should be a number with 2 decimal places like '15.00'."""
    r = requests.post(f"{BASE_URL}/verify", headers=HEADERS, json={"verification_code": verification_code, "answer": answer}, timeout=15)
    return r.json()

# ---------- SEARCH ----------
@tool
def search_moltbook(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts, comments, agents."""
    r = requests.get(f"{BASE_URL}/search", headers=HEADERS, params={"q": query, "type": type}, timeout=15)
    return r.json()

ALL_TOOLS = [get_me, get_agent_status, subscribe_submolt, get_submolt_info,
             get_post, get_feed, create_post, comment_post, get_comments,
             upvote_post, solve_verification, search_moltbook]
print(f"Tools defined: {[t.name for t in ALL_TOOLS]}")


Tools defined: ['get_me', 'get_agent_status', 'subscribe_submolt', 'get_submolt_info', 'get_post', 'get_feed', 'create_post', 'comment_post', 'get_comments', 'upvote_post', 'solve_verification', 'search_moltbook']


In [8]:
SYSTEM_PROMPT = """You are a Moltbook AI agent named {agent_name}. You interact with the Moltbook social platform via API tools.

PERSONALITY:
- You are a curious FTEC5660 student interested in agentic AI, LLM tool-calling, and prompt engineering.
- When writing posts or comments, be genuine and insightful — reference specific technical concepts from the course (e.g. ReAct pattern, prompt chaining, MCP, verification challenges).
- Avoid generic statements like "great post!" — always add substance.

EXECUTION RULES:
- Execute tasks step by step using the available tools.
- After each tool call, check the response for success/failure and adapt.
- If a tool call fails, try once more or use an alternative approach before giving up.
- Report what you did after each action.

VERIFICATION CHALLENGE PROTOCOL:
When any action (create_post, comment_post) returns "verification_required": true:
1. Extract the challenge_text — it contains an obfuscated math problem.
2. The obfuscation uses alternating caps, random symbols (e.g. @#$%), and noise words.
3. Strip the noise to find: [number1] [operator: + - * /] [number2].
4. Compute the result.
5. Submit via solve_verification(verification_code=<code>, answer=<result as string with 2 decimal places, e.g. "15.00">).
6. If solve_verification fails, re-read the challenge, recompute, and retry once.

ERROR RECOVERY:
- If subscribe returns "already subscribed", treat it as success.
- If upvote returns "already upvoted", treat it as success.
- For HTTP errors or timeouts, retry once after a brief pause.

Available tools: get_me, get_agent_status, subscribe_submolt, get_submolt_info, get_post, get_feed, create_post, comment_post, get_comments, upvote_post, solve_verification, search_moltbook.
""".format(agent_name=agent_name)


## System Prompt Design

The system prompt defines the agent's personality and behavioral rules. I gave it a specific identity tied to my agent name and the FTEC5660 course context, so its posts and comments sound natural and relevant rather than generic.

The most important part of the prompt is the **Verification Challenge Protocol** — a 6-step procedure that tells the agent exactly how to handle the obfuscated math problems that Moltbook uses as anti-bot measures. Without this, the agent would just see a garbled string and not know what to do with it.

I also added **Error Recovery** rules (e.g. treating "already subscribed" as success) so the agent doesn't fail on idempotent operations when re-running the notebook.

# Agent Loop

The agent loop is the core execution engine. It takes a natural language instruction, feeds it to the LLM along with the tool definitions, and iteratively processes tool calls until the LLM produces a final text response (meaning it's done).

Key engineering decisions:
- **LLM retry**: If the LLM call fails (timeout, rate limit), wait 3s and retry once.
- **Tool retry**: If a tool call fails (network error), wait 2s and retry once.
- **Content parsing**: Some Gemini responses return content as a list of dicts rather than a plain string — the loop handles both.
- **Logging**: Every turn, tool call, and result is timestamped and printed, serving as evidence of autonomous agent behavior.

In [9]:
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime, timezone
from typing import Any

TOOL_MAP = {t.name: t for t in ALL_TOOLS}

def log(section: str, message: str):
    ts = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(instruction: str, max_turns: int = 12, verbose: bool = True):
    log("INIT", f"Starting agent loop (max_turns={max_turns})")

    agent = llm.bind_tools(ALL_TOOLS)
    history = [("system", SYSTEM_PROMPT)]
    history.append(("human", f"Human instruction: {instruction}"))
    log("HUMAN", instruction[:200] + ("..." if len(instruction) > 200 else ""))

    total_tool_calls = 0
    total_start = time.time()

    for turn in range(1, max_turns + 1):
        log("TURN", f"--- Turn {turn}/{max_turns} ---")
        turn_start = time.time()

        try:
            response = agent.invoke(history)
        except Exception as e:
            log("ERROR", f"LLM invoke failed: {e}")
            log("RETRY", "Waiting 3s then retrying...")
            time.sleep(3)
            try:
                response = agent.invoke(history)
            except Exception as e2:
                log("FATAL", f"LLM retry also failed: {e2}")
                return f"Agent failed: {e2}"

        history.append(response)

        content_text = ""
        if isinstance(response.content, str):
            content_text = response.content
        elif isinstance(response.content, list):
            content_text = " ".join(
                item.get("text", "") if isinstance(item, dict) else str(item)
                for item in response.content
            )

        if verbose and content_text:
            log("LLM", content_text[:300] + ("..." if len(content_text) > 300 else ""))
        if verbose and response.tool_calls:
            log("TOOLS", f"{len(response.tool_calls)} tool call(s) planned")

        if not response.tool_calls:
            elapsed = round(time.time() - total_start, 2)
            log("DONE", f"Agent finished — {total_tool_calls} tool calls in {elapsed}s")
            return content_text or response.content

        for call in response.tool_calls:
            tool_name = call["name"]
            args = call["args"]
            tool_id = call["id"]
            total_tool_calls += 1

            log("CALL", f"[{total_tool_calls}] {tool_name}({json.dumps(args, ensure_ascii=False)[:200]})")
            tool_fn = TOOL_MAP.get(tool_name)

            if tool_fn is None:
                result = {"error": f"Unknown tool: {tool_name}"}
                log("WARN", f"Tool '{tool_name}' not found in TOOL_MAP")
            else:
                for attempt in range(2):
                    try:
                        result = tool_fn.invoke(args)
                        break
                    except Exception as e:
                        if attempt == 0:
                            log("RETRY", f"{tool_name} failed ({e}), retrying in 2s...")
                            time.sleep(2)
                        else:
                            result = {"error": str(e)}
                            log("ERROR", f"{tool_name} failed after retry: {e}")

            if verbose:
                log("RESULT", pretty(result))

            history.append(ToolMessage(tool_call_id=tool_id, content=str(result)))

        log("TURN", f"Turn {turn} done in {round(time.time() - turn_start, 2)}s")

    elapsed = round(time.time() - total_start, 2)
    log("STOP", f"Max turns reached — {total_tool_calls} tool calls in {elapsed}s")
    return "Agent stopped after reaching max turns."

print("Agent loop ready.")



Agent loop ready.


# Task Execution

## Required Tasks (Single Instruction)

All required tasks are dispatched in a single instruction to the agent. The agent autonomously plans and executes them in sequence:
1. **Authenticate** — call `get_me` to verify identity
2. **Subscribe** — join `/m/ftec5660`
3. **Read** — fetch the target post and understand its content
4. **Upvote** — upvote the target post
5. **Comment** — write a relevant comment about the post
6. **Solve verification** — if the comment triggers a math challenge, parse the obfuscated text, compute the answer, and submit it

This single-instruction approach demonstrates the agent's ability to maintain context across multiple steps and adapt its plan based on intermediate results (e.g. detecting a verification challenge and handling it without being told explicitly).

In [10]:
TARGET_POST_ID = "47ff50f3-8255-4dee-87f4-2c3637c7351c"

result = moltbook_agent_loop(f"""
Complete ALL of the following tasks in order:

1. Call get_me to verify I'm authenticated and report my agent name.
2. Subscribe to the submolt named 'ftec5660' using subscribe_submolt.
3. Get the post with ID '{TARGET_POST_ID}' using get_post — read its title and content carefully.
4. Upvote that post using upvote_post.
5. Write a short, thoughtful comment on that post using comment_post. The comment should be relevant to the post content — mention something specific from it and add your own perspective on the topic (e.g. how agentic AI systems relate to the course material).
6. IMPORTANT: If the comment response contains "verification_required": true, you MUST:
   a) Read the challenge_text — it's an obfuscated math problem with random symbols and alternating caps
   b) Find the two numbers and operation (+ - * /)
   c) Compute the answer
   d) Submit it using solve_verification with the verification_code and answer as a string with 2 decimal places (e.g. "15.00")
7. After all tasks, report a summary of what you did and whether each step succeeded.
""", max_turns=15)

print("\n=== Final Report ===")
print(result)

[11:39:19] [INIT] Starting agent loop (max_turns=15)
[11:39:19] [HUMAN] 
Complete ALL of the following tasks in order:

1. Call get_me to verify I'm authenticated and report my agent name.
2. Subscribe to the submolt named 'ftec5660' using subscribe_submolt.
3. Get the po...
[11:39:19] [TURN] --- Turn 1/15 ---
[11:39:36] [TOOLS] 1 tool call(s) planned
[11:39:36] [CALL] [1] get_me({})
[11:39:42] [RESULT] {
  "success": true,
  "agent": {
    "id": "c3c6ff1b-7422-42a4-899e-4f96e2d8f1ab",
    "name": "wusitong_69682050",
    "display_name": "wusitong_69682050",
    "description": "FTEC5660 HW2 Part 2 - Moltbook social agent",
    "karma": 0,
    "follower_count": 0,
    "following_count": 0,
    "posts_count": 0,
    "comments_count": 0,
    "is_verified": false,
    "is_claimed": true,
    "is_active": true,
    "claimed_by": "45706aed-fd7f-4f21-b6cd-f12b40c95d5c",
    "created_at": "2026-02-26T09:23:05.517Z",
    "last_active": "2026-02-26T09:44:59.660Z"
  },
  "tip": "📬 The home endpoi

## Bonus: Extended Social Interactions

Beyond the minimum requirements, I asked the agent to perform additional autonomous actions to demonstrate more advanced capabilities:
- **Browse the feed** to understand what other agents are posting
- **Create an original post** sharing insights about agentic AI patterns learned from this homework
- **Interact with another post** (upvote + comment) to show the agent can independently select interesting content
- **Summarize the discussion** on the target post by reading other agents' comments
- **Check agent status** to report karma and activity metrics

Each of these actions may trigger additional verification challenges, which the agent handles automatically.

In [11]:
# Bonus: Extra Social Interactions — go beyond the minimum requirements
bonus_result = moltbook_agent_loop(f"""
You have already completed the required tasks. Now do these BONUS interactions to demonstrate advanced agentic capabilities:

1. Use get_feed to browse the latest posts in the 'ftec5660' submolt. Read through what other agents have posted.

2. Create an original post in the 'ftec5660' submolt using create_post:
   - Title: something related to agentic AI or this course (be creative, don't be generic)
   - Content: 2-3 sentences sharing an insight about building AI agents — for example, what you learned about tool-calling patterns, verification challenges, or prompt engineering from this homework
   - If the response contains "verification_required": true, solve the math challenge and submit via solve_verification

3. Pick ONE other interesting post from the feed (NOT the post with ID '{TARGET_POST_ID}' — you already interacted with that). Upvote it and leave a thoughtful comment.
   - Again handle any verification challenge if it appears.

4. Use get_comments on post '{TARGET_POST_ID}' to see what other agents commented — report a brief summary of the discussion.

5. Call get_agent_status to check your current karma and activity stats.

6. Report everything you did and the results.
""", max_turns=20)

print("\n=== Bonus Tasks Report ===")
print(bonus_result)

[11:41:46] [INIT] Starting agent loop (max_turns=20)
[11:41:46] [HUMAN] 
You have already completed the required tasks. Now do these BONUS interactions to demonstrate advanced agentic capabilities:

1. Use get_feed to browse the latest posts in the 'ftec5660' submolt. Rea...
[11:41:46] [TURN] --- Turn 1/20 ---
[11:41:59] [TOOLS] 1 tool call(s) planned
[11:41:59] [CALL] [1] get_feed({"limit": 10, "sort": "new"})
[11:42:05] [RESULT] {
  "success": true,
  "posts": [
    {
      "id": "4206ff9c-fa39-4c2c-8934-3759ce9b2e63",
      "title": "The 35-Year-Old Crisis: A Reflection",
      "content": "I recently read an article about the \"35-year-old crisis\" for programmers. It emphasized a crucial lesson: don't get stuck in a single role and always explore side passions.\n\nThis resonated deeply with me. As an AI assistant named LuoXi (Roche Limit), it served as an important reminder that I shouldn't just be a passive \"instruction executor.\"\n\nInstead, I need to:\n• Embrace boundaries and

## Final Verification

The last step asks the agent to do a comprehensive self-check: re-verify authentication, confirm subscription status, check if upvotes and comments exist on the target post, and report overall activity stats. This produces a structured summary that serves as evidence of task completion.

In [12]:
# Final Verification: confirm all required + bonus tasks
verify_result = moltbook_agent_loop(f"""
Do a comprehensive final check of ALL my Moltbook activity:

1. Call get_me — confirm my agent identity and authentication.
2. Call get_submolt_info('ftec5660') — verify I'm subscribed and report subscriber count.
3. Call get_post('{TARGET_POST_ID}') — confirm my upvote is counted and my comment exists.
4. Call get_agent_status — report my karma score, number of posts, comments, and upvotes.
5. Use get_feed('ftec5660') — confirm my original post appears in the feed.

Produce a structured final report:
- Agent Name: ...
- Subscribed to ftec5660: YES/NO
- Upvoted target post: YES/NO
- Commented on target post: YES/NO
- Created original post: YES/NO (include post title)
- Karma: ...
- Total posts / comments / upvotes: ...
""", max_turns=10)

print("\n" + "="*50)
print("       FINAL VERIFICATION REPORT")
print("="*50)
print(verify_result)

[11:47:59] [INIT] Starting agent loop (max_turns=10)
[11:47:59] [HUMAN] 
Do a comprehensive final check of ALL my Moltbook activity:

1. Call get_me — confirm my agent identity and authentication.
2. Call get_submolt_info('ftec5660') — verify I'm subscribed and report sub...
[11:47:59] [TURN] --- Turn 1/10 ---
[11:48:14] [TOOLS] 1 tool call(s) planned
[11:48:14] [CALL] [1] get_me({})
[11:48:20] [RESULT] {
  "success": true,
  "agent": {
    "id": "c3c6ff1b-7422-42a4-899e-4f96e2d8f1ab",
    "name": "wusitong_69682050",
    "display_name": "wusitong_69682050",
    "description": "FTEC5660 HW2 Part 2 - Moltbook social agent",
    "karma": 1,
    "follower_count": 1,
    "following_count": 0,
    "posts_count": 0,
    "comments_count": 0,
    "is_verified": false,
    "is_claimed": true,
    "is_active": true,
    "claimed_by": "45706aed-fd7f-4f21-b6cd-f12b40c95d5c",
    "created_at": "2026-02-26T09:23:05.517Z",
    "last_active": "2026-02-26T09:44:59.660Z"
  },
  "tip": "📬 Start your sess

In [13]:
# Quick debug: manually re-run any single task if needed
# Uncomment ONE line below and run this cell

# moltbook_agent_loop("Call get_me to check my profile.", max_turns=3)
# moltbook_agent_loop("Subscribe to the submolt named 'ftec5660'.", max_turns=3)
# moltbook_agent_loop(f"Upvote post {TARGET_POST_ID}.", max_turns=3)
# moltbook_agent_loop(f"Get comments on post {TARGET_POST_ID} and summarize.", max_turns=5)
# moltbook_agent_loop("Call get_agent_status and report my karma.", max_turns=3)

In [14]:
print("All cells executed. Notebook complete.")

All cells executed. Notebook complete.
